# Error analysis (the 'diagnose' stage of EDD)

Review real traces, categorise failures, and turn the recurring ones into new golden cases. **Error-analysis-first**: write evals for errors you actually see here — not ones you imagine.

Generate traces first by asking questions via the CLI: `uv run python -m policy_copilot.cli "..."` (or run `agent.main()`). Then run this notebook.

In [ ]:
import pandas as pd

from policy_copilot.tracing import load_traces

df = pd.DataFrame(load_traces())
print(f"{len(df)} traces")
df.tail(10)

## 1. Quick health view

In [ ]:
if len(df):
    print("refused counts:")
    print(df["refused"].value_counts())
    print("\ntop_score summary:")
    print(df["top_score"].describe())

## 2. Candidates to inspect

Sort by retrieval score — weak retrieval and borderline refusals are where failures cluster.

In [ ]:
if len(df):
    cols = ["question", "refused", "top_score", "citations", "answer"]
    display(df.sort_values("top_score")[cols].head(10))

## 3. Diagnose → new golden cases

Read the rows above and label each failure mode. Suggested categories:
`retrieval-missed-table` · `number-paraphrased` · `should-have-refused` · `wrong-citation` · `ok`.

Count the categories, fix the most frequent cause (prompt / chunking / threshold), and append the failing questions to `evals/golden.jsonl` so the CI gate catches regressions next time.

In [ ]:
# Template: label by inspection, then tally.
if len(df):
    df["failure_mode"] = ""  # fill in for the rows you reviewed above
    # df.loc[df['question'].str.contains('revenue'), 'failure_mode'] = 'number-paraphrased'
    print(df["failure_mode"].value_counts())